# 缓存 & 去重

**常见用法**：同参重复查询短路（天气、汇率、搜索、报表），省真实调用与费用；
跨轮对话中相同问题的去重。

**钩子内的做法**：
- 缓存 key 用 `(工具名, str(args))`——args 是 dict 不可哈希，必须字符串化（稳定 key 用 `json.dumps(args, sort_keys=True)`）
- 命中 → **跳过 execute**，用当前 tool_call_id 重建 ToolMessage 返回（直接复用旧 ToolMessage 会带旧 id，协议错位）
- 未命中 → `execute(request)` 后**写缓存再返回**（只读不写 = 永远不命中，实测踩过）；进阶加 TTL

In [8]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


tool_use_cache: dict[tuple, str] = {}  # { (工具名, 参数) : 结果 }


def wrap_tool_call(request: ToolCallRequest, execute) -> ToolMessage | object:
    """按 (name, args) 查缓存，命中直接返回，不执行"""
    tc = request.tool_call
    key = (tc["name"], str(tc["args"]))

    if key in tool_use_cache:
        print(f"[缓存命中] {tc['name']}({tc['args']})")
        content = tool_use_cache[key]
    else:
        print(f"[缓存未命中] {tc['name']}({tc['args']})")
        content = execute(request).content
        tool_use_cache[key] = content
    return ToolMessage(content=content, name=tc["name"], tool_call_id=tc["id"])


# 工具节点
tool_node = ToolNode(tools, wrap_tool_call=wrap_tool_call)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: ChatState) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}
graph = builder.compile(checkpointer)
res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气,以及科技方面的新闻")]}, config=config)
print(res)

[缓存未命中] get_weather({'city': '北京'})

[缓存未命中] get_news({'topic': '科技'})

{
    'messages': [
        HumanMessage(
            content='帮我查一下北京的天气,以及科技方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='74d31b75-6964-4147-bded-2823741c5a8d'
        ),
        AIMessage(
            content='我来帮您查询北京的天气和科技新闻。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 76,
                    'prompt_tokens': 346,
                    'total_tokens': 422,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 218
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'a83c750d-d0b3-47df-9d0e-491e10271f79',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b03a-1689-7f02-b834-46ab42b91c24-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_9CL7QAGr8EFIrnoFiWOo2231',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_CSf3Vb1oUpcF7FxhI39C2710',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 346,
                'output_tokens': 76,
                'total_tokens': 422,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='北京 的天气是晴天，温度 25°C',
            name='get_weather',
            id='fc879e80-18cd-4cbb-8bc9-22b1aabf7fd9',
            tool_call_id='call_00_9CL7QAGr8EFIrnoFiWOo2231'
        ),
        ToolMessage(
            content='最新科技新闻：AI 技术正在快速发展。',
            name='get_news',
            id='f555e807-cfe2-460d-9a72-c37d5de02039',
            tool_call_id='call_01_CSf3Vb1oUpcF7FxhI39C2710'
        ),
        AIMessage(
            content='已经帮您查询好了，以下是结果：\n\n**🌤️ 北京天气**\n- 天气：晴天\n- 温度：25°C\n\n**📰 
科技新闻**\n- 最新科技新闻：AI 技术正在快速发展。\n\n今天北京天气晴朗、温度宜人，很适合外出活动。科技领域方面，AI 
技术依然是最受关注的热点。如果您想了解更详细的天气信息或其他方面的新闻，随时告诉我！',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 91,
                    'prompt_tokens': 463,
                    'total_tokens': 554,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 207
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '4a283dec-346c-4048-bdd5-1a9357258f4d',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0b03a-1a07-7090-8bad-d898e3008663-0',
            tool_calls=[],
            invalid_tool_calls=

In [9]:
print(tool_use_cache)

{
    ('get_weather', "{'city': '北京'}"): '北京 的天气是晴天，温度 25°C',
    ('get_news', "{'topic': '科技'}"): '最新科技新闻：AI 技术正在快速发展。'
}

In [10]:
res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气,以及体育方面的新闻")]}, config=config)
print(res)

[缓存命中] get_weather({'city': '北京'})

[缓存未命中] get_news({'topic': '体育'})

{
    'messages': [
        HumanMessage(
            content='帮我查一下北京的天气,以及科技方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='74d31b75-6964-4147-bded-2823741c5a8d'
        ),
        AIMessage(
            content='我来帮您查询北京的天气和科技新闻。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 76,
                    'prompt_tokens': 346,
                    'total_tokens': 422,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 218
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'a83c750d-d0b3-47df-9d0e-491e10271f79',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b03a-1689-7f02-b834-46ab42b91c24-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_9CL7QAGr8EFIrnoFiWOo2231',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '科技'},
                    'id': 'call_01_CSf3Vb1oUpcF7FxhI39C2710',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 346,
                'output_tokens': 76,
                'total_tokens': 422,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='北京 的天气是晴天，温度 25°C',
            name='get_weather',
            id='fc879e80-18cd-4cbb-8bc9-22b1aabf7fd9',
            tool_call_id='call_00_9CL7QAGr8EFIrnoFiWOo2231'
        ),
        ToolMessage(
            content='最新科技新闻：AI 技术正在快速发展。',
            name='get_news',
            id='f555e807-cfe2-460d-9a72-c37d5de02039',
            tool_call_id='call_01_CSf3Vb1oUpcF7FxhI39C2710'
        ),
        AIMessage(
            content='已经帮您查询好了，以下是结果：\n\n**🌤️ 北京天气**\n- 天气：晴天\n- 温度：25°C\n\n**📰 
科技新闻**\n- 最新科技新闻：AI 技术正在快速发展。\n\n今天北京天气晴朗、温度宜人，很适合外出活动。科技领域方面，AI 
技术依然是最受关注的热点。如果您想了解更详细的天气信息或其他方面的新闻，随时告诉我！',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 91,
                    'prompt_tokens': 463,
                    'total_tokens': 554,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 207
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '4a283dec-346c-4048-bdd5-1a9357258f4d',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0b03a-1a07-7090-8bad-d898e3008663-0',
            tool_calls=[],
            invalid_tool_calls=

In [11]:
print(tool_use_cache)

{
    ('get_weather', "{'city': '北京'}"): '北京 的天气是晴天，温度 25°C',
    ('get_news', "{'topic': '科技'}"): '最新科技新闻：AI 技术正在快速发展。',
    ('get_news', "{'topic': '体育'}"): '最新体育新闻：中国队取得了胜利。'
}